In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
antibiotic_cohort = pd.read_csv(f"{WORK_DIR}/microbiology_cultures_cohort.csv")
antibiotic_cohort = antibiotic_cohort[['anon_id',
                                       'pat_enc_csn_id_coded',
                                       'order_proc_id_coded',
                                       'order_time_jittered_utc']]
antibiotic_cohort

In [ ]:
procedure_df = pd.read_csv(f"{RAW_DIR}/clarity/2025-05-19.done/cpt_2025-05-19.csv")
procedure_df = procedure_df.rename(columns={'patientID':'anon_id',
                                             'patientEncounterID':'pat_enc_csn_id_coded',
                                             'cptCodeDT':'start_date_jittered_utc'})
procedure_df

In [ ]:
def filter_procedure(df, code_min, code_max, description):
    df_filtered = df.copy()
    df_filtered['code'] = pd.to_numeric(df_filtered['cptCD'], errors='coerce')
    return df_filtered[
        (df_filtered['code'] >= code_min) &
        (df_filtered['code'] <= code_max)
    ][['anon_id', 'start_date_jittered_utc']].drop_duplicates().assign(
        procedure_description=description,
        procedure_time=lambda x: x['start_date_jittered_utc']
    )

In [ ]:
dialysis = filter_procedure(procedure_df, 90935, 90999, 'dialysis')
cvc = filter_procedure(procedure_df, 36555, 36573, 'cvc')
urethral_catheter = filter_procedure(procedure_df, 51701, 51703, 'urethral_catheter')

In [ ]:
procedure_surgery = pd.read_csv(f'{RAW_DIR}/edw/procedures/MGB_procedures_20240715.csv')
surgical_CPT = procedure_surgery['CPT'].unique()

In [ ]:
surgical_procedures = procedure_df.copy()
surgical_procedures = surgical_procedures[surgical_procedures['cptCD'].isin(surgical_CPT)][['anon_id','start_date_jittered_utc']].drop_duplicates().assign(procedure_description='surgical_procedure',
        procedure_time=lambda x: x['start_date_jittered_utc'])
surgical_procedures


In [ ]:
ventilation = pd.read_csv(f'{RAW_DIR}/clarity/2025-05-19.done/o2_devices_2025-05-19.csv')

In [ ]:
mechvent = ventilation[ventilation['o2DeviceMappedv2'].isin(['VENTILATOR'])]
mechvent = mechvent[['patientID','recordedDTS']].rename(columns={'patientID':'anon_id',
                                                                 'recordedDTS':'procedure_time'}).drop_duplicates()
mechvent['procedure_description'] = 'mechvent'
mechvent

In [ ]:
## Parenteral Nutrition

In [ ]:
procedure_frames = [
    dialysis,
    cvc,
    mechvent,
    surgical_procedures,
    # parenteral_nutrition,
    urethral_catheter
]

all_procedure = pd.concat(procedure_frames, ignore_index=True)

# Step 7: Join with antibiotic cohort
merged = antibiotic_cohort.merge(all_procedure, on='anon_id', how='inner')

merged

In [ ]:
merged = merged.dropna(subset=['procedure_time'])
merged = merged.drop_duplicates()
merged

In [ ]:
# Step 8: Calculate time difference (only keep where diff > 0)
merged['order_time_jittered_utc'] = pd.to_datetime(merged['order_time_jittered_utc']).dt.tz_localize(None)
merged['procedure_time'] = pd.to_datetime(merged['procedure_time'], utc=True).dt.tz_localize(None)
merged['procedure_days_culture'] = (
    (merged['order_time_jittered_utc'] - merged['procedure_time']).dt.days
)

final_df = merged[
    merged['procedure_days_culture'] > 0
].drop_duplicates()

# Optional: sort and group if needed
final_df = final_df.sort_values(
    by=['anon_id', 'order_proc_id_coded', 'pat_enc_csn_id_coded', 
        'order_time_jittered_utc', 'procedure_description', 'procedure_time']
)

final_df

In [ ]:
final_df = final_df[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded',
                     'order_time_jittered_utc','procedure_description', 'procedure_days_culture']]
final_df = final_df.drop_duplicates()
final_df

In [ ]:
final_df.to_csv(f'{WORK_DIR}/microbiology_cultures_priorprocedures.csv',
                index = False)

## Deidentification

In [ ]:
# Load the saved mappings
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = final_df.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_priorprocedures.csv',
                        index = False)